# Pipeline da Base — TCC Pair Trading

Notebook único que regenera **todas** as bases de preço do projeto, do `Pt.csv` do
orientador até os dois arquivos finais.

| Parte | Saída |
|---|---|
| 1 | `professor_consolidado.csv` — bloco do orientador 1990/S2–2015/S2, com datas e correções |
| 2 | `extensao_2016_2025.csv` — extensão própria 2016/S1–2025/S2 |
| 3 | `base_completa.csv` — concatenação direta (**padrão**, com o salto na fronteira) |
| 4 | `base_unificada.csv` — extensão emendada por retornos (**recomendada**, sem salto) |
| 5 | Validação numérica das duas |

As duas bases finais têm estrutura idêntica e são intercambiáveis em `pairs_formation` /
`pairs_trading`. A diferença é só a fronteira 2015→2016: `base_completa` preserva os
`adj_close` como vieram de cada fonte; `base_unificada` reescala a extensão para emendar
no nível de preço do orientador, preservando todos os retornos diários.

**Ordem:** rode de cima para baixo. Só a Parte 2.2 (download) precisa de internet, e ela
está desligada por padrão — os CSVs de `prices/` já estão no repositório.

Diagnósticos e contrafactuais que embasam cada correção: [`src/validation/`](../validation/).

## 0. Setup e flags

Todas as decisões de construção ficam aqui. Cada flag é independente e reversível.

In [110]:
from pathlib import Path
import numpy as np
import pandas as pd

# ── Caminhos ──────────────────────────────────────────────────────────────────
PROFESSOR_DIR = Path("../../data_bases/professor")
OUTPUT_DIR    = Path("../../data_bases")
EXTERNAL_DIR  = Path("../../data_bases/external")
PRICES_YAHOO  = Path("../../data_bases/prices")
PRICES_TIINGO = Path("../../data_bases/prices_tiingo")
PRICES_SOBR   = Path("../../data_bases/prices_sobreposição")

META_COLS = ["year", "semester", "period", "day_in_semester", "total_days_sem"]

# ── Parte 1: correções do bloco do professor ─────────────────────────────────
# Cinco séries do Pt.csv estão deslocadas um pregão em relação às outras ~1.095.
# Confirmado contra fonte externa: SPGI (Yahoo, corr 0,958 em D−1), MRO (Tiingo, 0,9999
# em D−1), KATE (Tiingo, 0,986 em D−1), BF_B (Yahoo BF-B, 0,974 em D+1). ARB não tem
# fonte, mas tem a mesma assinatura contra o mercado. Nove controles batem em D com 0,99+.
# Sem esta correção, MHFI/SPGI e BF_B/1958Q viram "a mesma empresa contra si mesma" e
# respondem por 68% do P&L pré-2016.  → src/validation/por_que_2_pares_dominam.md
CORRIGIR_SERIES_DESLOCADAS = True

# 8.561 células com preço exatamente 0 (USW, 571300Q, FBNIQ, QCOM). Zero não é preço:
# gera retorno de −100% e +inf quando a série volta.
PRECO_ZERO_VIRA_NAN = True

# O Pt.csv tem 6.426 linhas, mas Periods.csv soma 6.425 (conta dias de RETORNO, não de
# preço). A versão anterior fazia min(6425, 6426) e descartava a última linha — o pregão
# de 2015-12-31. True recupera esse dia.  → src/validation/revisao_completa_2026-09-13.md §2.4
RECUPERAR_ULTIMO_PREGAO = True

# ⚠️ MÁSCARA DE MEMBERSHIP DO ÍNDICE (ticker_sem.csv)
# O Pt.csv NÃO está mascarado: traz 612–836 séries por semestre, contra ~500 membros.
# Aplicar a máscara é defensável como método, mas NÃO reproduz o artigo do orientador:
#
#            universo   ret % a.a.   Sharpe
#   artigo     ~500        3,600      0,703
#   False       707        3,015      0,705   ← reproduz
#   True        467        1,292      0,263
#
# Medido: a queda não é remoção de viés — um corte ALEATÓRIO do mesmo tamanho dá
# 0,92% / 2,04% / 3,08% em três sorteios, e 1,292 cai dentro dessa faixa.
# → src/validation/revisao_completa_2026-09-13.md §1, §1.1  ·  controle_universo.py
APLICAR_MASCARA_INDICE = True

# ── Parte 2: extensão ─────────────────────────────────────────────────────────
# True tenta baixar do Yahoo os tickers sem arquivo em prices/ ou prices_tiingo/.
# Deixe False para rodar offline — os CSVs já estão no repositório.
BAIXAR_FALTANTES = False

# APC e INFO vieram do financialdata.net (gathering.ipynb, Parte J), única fonte do
# projeto que carimba o preço em data de CALENDÁRIO e não de pregão: 224 e 279 datas
# fora do pregão, das quais 193 e 241 são domingos.
CALENDARIO_QUEBRADO = {"APC", "INFO"}

# ── Parte 4: emenda da fronteira ──────────────────────────────────────────────
# Retorno máximo tolerado no primeiro dia da extensão. Acima disso a âncora está em outra
# convenção (spin-off, troca de ticker recuperada de fonte diferente) e cai para o
# fallback de retorno zero — fabricado, mas limitado. Ver Parte 4 e validação 5.3.
LIMITE_SALTO_FRONTEIRA = 0.25

# ── Parte 1 e 2: trechos de preço interpolados ───────────────────────────────
# Alguns trechos do Pt.csv não são preço negociado: a taxa de crescimento por dia
# de CALENDÁRIO fica exatamente constante por semanas, inclusive atravessando os
# fins de semana. É uma reta traçada entre dois pontos. O caso extremo é o `PSX`,
# que sobe +0,0000790 por dia corrido de jan/1995 a abr/2012 — e a Phillips 66 só
# passou a existir em 2012, quando foi cindida da ConocoPhillips.
#
# Por que atrapalha: o método da distância procura duas trajetórias que andam
# juntas, e duas retas andam juntas perfeitamente. `GIDL` e `MCAI`, empresas sem
# nenhuma relação, correlacionam 0,99998 em retornos só por serem ambas rampas.
#
# Escopo: 47 séries do Pt.csv (18.740 células, 0,40%) e 2 da extensão (273 células).
# Calibração: 14 blue chips de controle (XOM, MSFT, KO, JNJ, CAT, DUK, AEP, SO,
# XEL, GE, PG, MMM, BA, CVX) dão ZERO marcações.
# → src/validation/revisao_completa_2026-09-13.md § 4.3
REMOVER_TRECHOS_INTERPOLADOS = True
INTERP_MIN_RUN = 10      # pregões consecutivos com a mesma taxa para marcar o trecho
INTERP_RTOL    = 1e-3    # tolerância relativa na comparação das taxas

print("Flags:")
for _k in ["CORRIGIR_SERIES_DESLOCADAS", "PRECO_ZERO_VIRA_NAN", "RECUPERAR_ULTIMO_PREGAO",
           "APLICAR_MASCARA_INDICE", "BAIXAR_FALTANTES", "LIMITE_SALTO_FRONTEIRA",
           "REMOVER_TRECHOS_INTERPOLADOS", "INTERP_MIN_RUN"]:
    print(f"  {_k:<28} = {globals()[_k]}")

Flags:
  CORRIGIR_SERIES_DESLOCADAS   = True
  PRECO_ZERO_VIRA_NAN          = True
  RECUPERAR_ULTIMO_PREGAO      = True
  APLICAR_MASCARA_INDICE       = True
  BAIXAR_FALTANTES             = False
  LIMITE_SALTO_FRONTEIRA       = 0.25
  REMOVER_TRECHOS_INTERPOLADOS = True
  INTERP_MIN_RUN               = 10


---
# Parte 1 — Bloco do orientador (1990/S2 – 2015/S2)

Entrada: `professor/Pt.csv` (preços, sem coluna de data) e `professor/Periods.csv`
(dias por semestre). Saída: `professor_consolidado.csv`.

## 1.1 Estrutura dos arquivos

Três fatos verificados em [`src/validation/valida_estrutura_pt.py`](../validation/valida_estrutura_pt.py),
e que determinam todo o resto desta parte:

1. **`Rt.csv` é exatamente `pct_change(Pt.csv)`** — 4.728.757 células, erro máximo 1e-12.
   A convenção de retorno do orientador é a mesma que usamos.
2. **`Periods.csv` conta dias de RETORNO**, não de preço: soma 6.425, que é
   `len(Rt) = len(Rf) = len(Rm)`. O `Pt.csv` tem 6.426 linhas — a primeira é a âncora `P₀`.
3. **A aritmética da Tabela 4 do artigo fecha na unidade:**
   `6.425 − (127 + 125) = 6.173 observações`, `51 − 2 = 49 períodos de negociação`.

In [111]:
pt      = pd.read_csv(PROFESSOR_DIR / "Pt.csv")
periods = pd.read_csv(PROFESSOR_DIR / "Periods.csv", header=None)[0].astype(int)

n_rows = len(pt) if RECUPERAR_ULTIMO_PREGAO else int(periods.sum())

print(f"Pt.csv      : {len(pt):,} linhas × {pt.shape[1]:,} tickers")
print(f"Periods.csv : {len(periods)} semestres, {periods.sum():,} dias de RETORNO")
print(f"              (len(Pt) = {periods.sum():,} retornos + 1 âncora = {periods.sum() + 1:,})")
print(f"Linhas a usar: {n_rows:,}")
print()
print(f"Tabela 4 do artigo: {periods.sum()} − ({periods[0]} + {periods[1]}) = "
      f"{periods.sum() - periods[:2].sum():,} observações, {len(periods) - 2} períodos")

Pt.csv      : 6,426 linhas × 1,100 tickers
Periods.csv : 51 semestres, 6,425 dias de RETORNO
              (len(Pt) = 6,425 retornos + 1 âncora = 6,426)
Linhas a usar: 6,426

Tabela 4 do artigo: 6425 − (127 + 125) = 6,173 observações, 49 períodos


## 1.2 Datas reais de cada linha

O `Pt.csv` não tem coluna de data. Usamos o histórico do MSFT no Yahoo como calendário
real da NYSE.

```
pregões NYSE 1990-07-02 .. 2015-12-31   = 6.427
menos 1993-02-22 (ausente no Pt.csv)    = 6.426 = len(Pt)   ✓
```

Verificado contra o `adj_close` do Yahoo para 10 blue chips: erro relativo mediano de
**5 × 10⁻⁶** em 1990–92 e ~10⁻⁵–10⁻⁴ no resto, **sem deslocamento residual em nenhum
período**. A conferência roda na célula 1.3.

In [112]:
import yfinance as yf

_raw = yf.download("MSFT", start="1990-06-25", end="2016-01-05",
                   auto_adjust=True, progress=False)
msft_yahoo = _raw[("Close", "MSFT")] if isinstance(_raw.columns, pd.MultiIndex) else _raw["Close"]
msft_yahoo.index = pd.to_datetime(msft_yahoo.index).tz_localize(None)

PT_INICIO        = pd.Timestamp("1990-07-02")
PT_FIM           = pd.Timestamp("2015-12-31")
DIAS_AUSENTES_PT = [pd.Timestamp("1993-02-22")]

_cal = msft_yahoo.index[(msft_yahoo.index >= PT_INICIO) & (msft_yahoo.index <= PT_FIM)]
_cal = _cal[~_cal.isin(DIAS_AUSENTES_PT)]
true_dates = _cal[-n_rows:]

print(f"Pregões {PT_INICIO.date()}..{PT_FIM.date()} : {len(_cal) + len(DIAS_AUSENTES_PT):,}")
print(f"Menos {[str(d.date()) for d in DIAS_AUSENTES_PT]}      : {len(_cal):,}")
print(f"Linhas do Pt.csv                     : {len(pt):,}")
assert len(_cal) == len(pt), "contagem de pregões não bate com o Pt.csv"
print(f"\nDatas atribuídas: {true_dates[0].date()} → {true_dates[-1].date()} ({len(true_dates):,} linhas)")

# ── Rótulo de semestre ────────────────────────────────────────────────────────
# Periods conta RETORNOS: os periods[k] retornos do semestre k são as linhas de preço
# bnd[k]+1 .. bnd[k+1]. A linha 0 (âncora P₀) pertence ao primeiro semestre.
_lab, _y, _s = [], 1990, 2
for _n in periods:
    _lab += [f"{_y}/S{_s}"] * int(_n)
    _y, _s = (_y + 1, 1) if _s == 2 else (_y, 2)
_lab = (["1990/S2"] + _lab)[:len(true_dates)]

date_map = pd.DataFrame({"date": true_dates, "period": _lab})
date_map["year"]     = date_map["period"].str[:4].astype(int)
date_map["semester"] = date_map["period"].str[-1].astype(int)
date_map["day_in_semester"] = date_map.groupby("period").cumcount() + 1
date_map["total_days_sem"]  = date_map.groupby("period")["period"].transform("size")

pt_consolidado = pd.concat(
    [date_map[["date"] + META_COLS].reset_index(drop=True), pt.iloc[:n_rows].reset_index(drop=True)],
    axis=1,
).set_index("date")

DATA_ANCORA = pt_consolidado.index[-1]      # último pregão do bloco do professor

_d = (date_map["period"] == "2001/S2").sum()
print(f"2001/S2 → {_d} dias (esperado 123 retornos + 0/1 âncora) {'OK' if _d in (123, 124) else 'CONFERIR'}")
print(f"Base consolidada: {pt_consolidado.shape}  |  âncora da fronteira: {DATA_ANCORA.date()}")

Pregões 1990-07-02..2015-12-31 : 6,427
Menos ['1993-02-22']      : 6,426
Linhas do Pt.csv                     : 6,426

Datas atribuídas: 1990-07-02 → 2015-12-31 (6,426 linhas)
2001/S2 → 123 dias (esperado 123 retornos + 0/1 âncora) OK
Base consolidada: (6426, 1105)  |  âncora da fronteira: 2015-12-31


## 1.3 Conferência do alinhamento

Compara os preços do `Pt.csv` com o `adj_close` do Yahoo de hoje. O orientador coletou em
~dez/2015, então a razão `professor / adj_close_hoje` é o fator de dividendos pagos desde
2016 — **constante por ticker**. Se as datas estiverem certas, o erro relativo dessa razão
é ~0 em todos os períodos.

In [113]:
_TK = ["MSFT", "JNJ", "KO", "XOM", "PG", "GE", "MRK", "PEP", "MMM", "BA"]
_yh = yf.download(_TK, start="1990-06-25", end="2016-01-05", auto_adjust=True, progress=False)["Close"]
_yh.index = pd.to_datetime(_yh.index).tz_localize(None)

_P = pt_consolidado[_TK].values
_Y = _yh[_TK].reindex(pt_consolidado.index).values
_fac = np.nanmedian(_P[:600] / _Y[:600], axis=0)          # fator de ajuste 2015→hoje
_err = pd.Series(np.nanmedian(np.abs(_P / _fac / _Y - 1), axis=1), index=pt_consolidado.index)

print("Erro relativo mediano por período (alinhado ⇒ ~0):")
for _lb, _m in [("1990-92", _err.index.year <= 1992),
                ("1993",    _err.index.year == 1993),
                ("1994-00", (_err.index.year >= 1994) & (_err.index.year <= 2000)),
                ("2001-15", _err.index.year >= 2001)]:
    print(f"  {_lb:<9} n={_m.sum():>5}  err={np.nanmedian(_err[_m]):.6f}")
print("\nTodos < 1e-3 ⇒ mapeamento de datas correto.")

Erro relativo mediano por período (alinhado ⇒ ~0):
  1990-92   n=  634  err=0.000005
  1993      n=  252  err=0.000004
  1994-00   n= 1767  err=0.000022
  2001-15   n= 3773  err=0.000076

Todos < 1e-3 ⇒ mapeamento de datas correto.


## 1.4 Correções de qualidade

| Defeito | Escopo | Correção |
|---|---|---|
| 5 séries deslocadas um pregão entre si | `SPGI`, `MRO`, `KATE`, `ARB` (atrasadas), `BF_B` (adiantada) | `shift` por coluna |
| Preço zero tratado como dado válido | 8.561 células em 4 tickers | `0 → NaN` |
| **Trechos de preço interpolados** | 18.740 células em 47 séries | `trecho → NaN` |

As quatro atrasadas têm o valor do pregão **anterior** carimbado no dia D, então
`valor_correto(D) = arquivo(D+1)` → `shift(-1)`. `BF_B` é o contrário → `shift(+1)`.
O shift deixa uma ponta sem valor; para os tickers que seguem na extensão essa ponta é a
âncora da emenda da Parte 4, e é preenchida com o retorno real de `prices_sobreposição/`.
**Trechos interpolados.** Parte do `Pt.csv` não é preço negociado: a taxa de crescimento
por dia de **calendário** fica exatamente constante por semanas, atravessando os fins de
semana (a variação de sexta para segunda é 3× a de um dia). É uma reta traçada entre dois
pontos. O `PSX` sobe +0,0000790 por dia corrido de jan/1995 a abr/2012 — e a Phillips 66
só existe desde 2012. Duas rampas dessas pareiam perfeitamente entre si: `GIDL` e `MCAI`,
empresas sem relação, correlacionam **0,99998** em retornos.

Removemos o **trecho**, não a série — o `PSX` mantém o histórico real de 2012 em diante.


In [114]:
_cols_preco = [c for c in pt_consolidado.columns if c not in META_COLS]

# ── (a) Séries deslocadas ────────────────────────────────────────────────────
DESLOC_ATRASADAS  = ["SPGI", "MRO", "KATE", "ARB"]   # valor de D é o do pregão anterior
DESLOC_ADIANTADAS = ["BF_B"]                          # valor de D é o do pregão seguinte

if CORRIGIR_SERIES_DESLOCADAS:
    _R   = pt_consolidado[_cols_preco].pct_change(fill_method=None)
    _mkt = _R.loc[:, _R.notna().sum() > 250].median(axis=1)
    _antes = {t: (_R[t].corr(_mkt.shift(1)), _R[t].corr(_mkt), _R[t].corr(_mkt.shift(-1)))
              for t in DESLOC_ATRASADAS + DESLOC_ADIANTADAS if t in pt_consolidado.columns}

    _rel = []
    for _t, _passo in [(t, -1) for t in DESLOC_ATRASADAS] + [(t, +1) for t in DESLOC_ADIANTADAS]:
        if _t not in pt_consolidado.columns:
            continue
        _prim, _ult = pt_consolidado[_t].first_valid_index(), pt_consolidado[_t].last_valid_index()
        pt_consolidado[_t] = pt_consolidado[_t].shift(_passo)
        _borda = _ult if _passo == -1 else _prim
        _trato = "NaN (borda sem valor recuperável)"

        _f = PRICES_SOBR / f"{_t}.csv"
        if _passo == -1 and _f.exists():
            _d = pd.read_csv(_f, index_col=0, parse_dates=[0]).iloc[:, 0]
            _pos = pt_consolidado.index.get_loc(_borda)
            _ant = pt_consolidado.index[_pos - 1]
            if (_borda in _d.index and _ant in _d.index
                    and pd.notna(pt_consolidado.at[_ant, _t]) and _d.loc[_ant] != 0):
                pt_consolidado.at[_borda, _t] = pt_consolidado.at[_ant, _t] * (_d.loc[_borda] / _d.loc[_ant])
                _trato = f"preenchida com retorno de sobreposição ({pt_consolidado.at[_borda, _t]:.4f})"
        _rel.append({"ticker": _t, "sentido": "atrasada (shift -1)" if _passo == -1 else "adiantada (shift +1)",
                     "borda": str(_borda.date()), "tratamento": _trato})

    print("=== (a) Séries deslocadas ===")
    print(pd.DataFrame(_rel).to_string(index=False))

    _Rd = pt_consolidado[_cols_preco].pct_change(fill_method=None)
    print(f"\n{'ticker':<8}{'':>8}{'mkt(D-1)':>11}{'mkt(D)':>11}{'mkt(D+1)':>11}  pico")
    for _t, _v0 in _antes.items():
        _s  = _Rd[_t]
        _v1 = (_s.corr(_mkt.shift(1)), _s.corr(_mkt), _s.corr(_mkt.shift(-1)))
        for _rot, _v in [("antes", _v0), ("depois", _v1)]:
            _p = ["D-1", "D", "D+1"][int(np.nanargmax(_v))]
            print(f"{_t if _rot == 'antes' else '':<8}{_rot:>8}{_v[0]:>11.4f}{_v[1]:>11.4f}{_v[2]:>11.4f}  {_p}")
    print("Esperado: pico em D-1/D+1 antes e em D depois (beta típico das sadias ≈ 0,46).")
else:
    print("(a) Correção de séries deslocadas DESLIGADA.")

# ── (b) Preço zero = ausência de dado ────────────────────────────────────────
if PRECO_ZERO_VIRA_NAN:
    _z = (pt_consolidado[_cols_preco] == 0)
    _n = int(_z.sum().sum())
    if _n:
        _pt = _z.sum(); _pt = _pt[_pt > 0].sort_values(ascending=False)
        _rel = []
        for _t, _q in _pt.items():
            _dz = pt_consolidado.index[pt_consolidado[_t] == 0]
            _depois = int((pt_consolidado[_t].loc[_dz[-1]:] > 0).sum())
            _rel.append({"ticker": _t, "zeros": int(_q), "de": str(_dz[0].date()), "ate": str(_dz[-1].date()),
                         "tipo": "final (delisting)" if _depois == 0 else "ISOLADO no meio da série"})
        pt_consolidado[_cols_preco] = pt_consolidado[_cols_preco].replace(0.0, np.nan)
        print(f"\n=== (b) Preço zero → NaN: {_n:,} células em {len(_pt)} tickers ===")
        print(pd.DataFrame(_rel).to_string(index=False))
    else:
        print("\n=== (b) Nenhum preço zero ===")
else:
    print("\n(b) Conversão de zeros DESLIGADA.")


# ── (c) Trechos de preço interpolados ────────────────────────────────────────
# Detector: taxa de crescimento por dia de CALENDÁRIO constante por INTERP_MIN_RUN
# pregões seguidos. Preço negociado nunca faz isso; uma interpolação linear em log
# sobre o calendário faz sempre, e a assinatura atravessa o fim de semana (a
# variação de sexta para segunda é exatamente 3x a de um dia).
def trechos_interpolados(serie, min_run=None, rtol=None):
    """Máscara booleana marcando os pregões que caem num trecho interpolado."""
    min_run = INTERP_MIN_RUN if min_run is None else min_run
    rtol    = INTERP_RTOL    if rtol    is None else rtol
    s = serie.dropna()
    if len(s) < min_run + 1:
        return pd.Series(False, index=serie.index)
    dt = s.index.to_series().diff().dt.days.values[1:].astype(float)
    g  = np.log(s.values[1:] / s.values[:-1]) / np.where(dt > 0, dt, np.nan)
    igual = np.isclose(g[1:], g[:-1], rtol=rtol, atol=1e-12)
    m, i = np.zeros(len(s), bool), 0
    while i < len(igual):
        if igual[i]:
            j = i
            while j < len(igual) and igual[j]:
                j += 1
            if (j - i + 2) >= min_run:          # nº de pregões no trecho
                m[i:j + 2] = True
            i = j
        else:
            i += 1
    return pd.Series(m, index=s.index).reindex(serie.index, fill_value=False)


if REMOVER_TRECHOS_INTERPOLADOS:
    _rel, _tot = [], 0
    for _t in _cols_preco:
        _m = trechos_interpolados(pt_consolidado[_t])
        _k = int(_m.sum())
        if _k:
            _d = pt_consolidado.index[_m.values]
            _rel.append({"ticker": _t, "obs": int(pt_consolidado[_t].notna().sum()),
                         "marcados": _k, "de": str(_d[0].date()), "ate": str(_d[-1].date())})
            pt_consolidado.loc[_m.values, _t] = np.nan
            _tot += _k
    _r = pd.DataFrame(_rel).sort_values("marcados", ascending=False)
    print()
    print(f"=== (c) Trechos interpolados → NaN: {_tot:,} células em {len(_r)} séries ===")
    print(_r.head(12).to_string(index=False))
    if len(_r) > 12:
        print(f"   ... mais {len(_r) - 12} séries (lista completa em _r)")
    print()
    print("Controles (devem ficar em 0):",
          {t: int(trechos_interpolados(pt_consolidado[t]).sum())
           for t in ["XOM", "MSFT", "KO", "JNJ", "CAT"] if t in _cols_preco})
else:
    print("\n(c) Remoção de trechos interpolados DESLIGADA.")


=== (a) Séries deslocadas ===
ticker              sentido      borda                                       tratamento
  SPGI  atrasada (shift -1) 2015-12-31 preenchida com retorno de sobreposição (97.2772)
   MRO  atrasada (shift -1) 2015-12-31 preenchida com retorno de sobreposição (12.5900)
  KATE  atrasada (shift -1) 2015-12-31                NaN (borda sem valor recuperável)
   ARB  atrasada (shift -1) 2013-09-30                NaN (borda sem valor recuperável)
  BF_B adiantada (shift +1) 1990-07-02                NaN (borda sem valor recuperável)

ticker             mkt(D-1)     mkt(D)   mkt(D+1)  pico
SPGI       antes     0.5998    -0.0113    -0.0562  D-1
          depois     0.0193     0.5999    -0.0113  D
MRO        antes     0.5502    -0.0141    -0.0333  D-1
          depois    -0.0106     0.5501    -0.0141  D
KATE       antes     0.5007     0.0309    -0.0042  D-1
          depois     0.0336     0.5007     0.0309  D
ARB        antes     0.3898     0.0159    -0.0088  D-1
      

## 1.5 Máscara de membership do índice

O `Pt.csv` **não** está mascarado: 612–836 séries por semestre contra ~500 membros do
índice. O orientador fornece a composição exata em `ticker_sem.csv` (502 × 52, uma coluna
por semestre).

> ⚠️ **Ligar esta máscara quebra a reprodução do artigo.** O artigo declara a amostra como
> as 1.100 séries do `Pt.csv` e o único filtro de universo que descreve é *"only stocks
> that are listed during the formation period are included"* — que é o `dropna` da janela
> de formação. Medido no método da distância (Top 20, committed capital, Jul/1991–Dez/2015):
>
> | | universo | ret % a.a. | Sharpe |
> |---|---|---|---|
> | artigo (Tabela 4) | ~500 | 3,600 | 0,703 |
> | máscara **desligada** | 707 | **3,015** | **0,705** |
> | máscara ligada | 467 | 1,292 | 0,263 |
>
> E a queda **não é remoção de viés**: um corte aleatório do mesmo tamanho dá 0,92%, 2,04%
> e 3,08% em três sorteios — 1,292 cai dentro dessa faixa. O efeito é o de ter menos
> candidatos, não o de limpar a amostra.
> Ver [`revisao_completa_2026-09-13.md`](../validation/revisao_completa_2026-09-13.md) §1 e §1.1.

Mantida como flag: ligar é uma escolha metodológica defensável (só negociar constituintes),
desde que declarada no texto — mas então os números não são confrontáveis com a Tabela 4.

In [115]:
if APLICAR_MASCARA_INDICE:
    _ts = pd.read_csv(PROFESSOR_DIR / "ticker_sem.csv", header=None)
    _sem_unicos = pt_consolidado["period"].drop_duplicates().tolist()
    if len(_sem_unicos) > _ts.shape[1]:
        raise ValueError(f"ticker_sem tem {_ts.shape[1]} colunas para {len(_sem_unicos)} semestres")

    _antes = int(pt_consolidado[_cols_preco].notna().sum().sum())
    _rel = []
    for _k, _per in enumerate(_sem_unicos):
        _memb   = set(_ts[_k].dropna()) & set(_cols_preco)
        _linhas = pt_consolidado["period"] == _per
        _fora   = [c for c in _cols_preco if c not in _memb]
        _rel.append({"semestre": _per, "membros": len(_memb),
                     "excedente_removido": int(pt_consolidado.loc[_linhas, _fora].notna().any().sum()),
                     "membro_sem_dado": int((~pt_consolidado.loc[_linhas, sorted(_memb)].notna().any()).sum())})
        pt_consolidado.loc[_linhas, _fora] = np.nan
    _depois = int(pt_consolidado[_cols_preco].notna().sum().sum())
    _r = pd.DataFrame(_rel)
    print("=== Máscara de membership LIGADA ===")
    print(f"Observações: {_antes:,} → {_depois:,}  ({_depois / _antes - 1:+.1%})")
    print(f"Excedente removido por semestre: min {_r.excedente_removido.min()}, "
          f"máx {_r.excedente_removido.max()}, média {_r.excedente_removido.mean():.0f}")
    print(f"Membros sem dado no Pt.csv: {int(_r.membro_sem_dado.sum())} de {int(_r.membros.sum()):,} ticker-semestres")
else:
    print("Máscara de membership DESLIGADA — universo = todas as séries do Pt.csv")
    print("(é a configuração que reproduz a Tabela 4 do artigo do orientador)")

_por_dia = pt_consolidado[_cols_preco].notna().sum(axis=1)
print(f"\nSéries com preço por dia: min {_por_dia.min()}, máx {_por_dia.max()}, mediana {_por_dia.median():.0f}")

pt_consolidado.to_csv(OUTPUT_DIR / "professor_consolidado.csv")
print(f"Salvo: professor_consolidado.csv  {pt_consolidado.shape}")

=== Máscara de membership LIGADA ===
Observações: 4,711,111 → 3,189,382  (-32.3%)
Excedente removido por semestre: min 107, máx 356, média 243
Membros sem dado no Pt.csv: 23 de 25,494 ticker-semestres

Séries com preço por dia: min 478, máx 502, mediana 497
Salvo: professor_consolidado.csv  (6426, 1105)


---
# Parte 2 — Extensão própria (2016/S1 – 2025/S2)

Preços `adj_close` coletados do Yahoo (`auto_adjust=True`) e do Tiingo (`adjClose`),
alinhados no calendário real de pregão e mascarados pela composição histórica do índice.

> **Tipo de preço.** O orientador usa `adj_close` coletado em ~dez/2015; a extensão usa
> `adj_close` coletado em 2026. É o mesmo tipo de preço, com data de normalização
> diferente — o que a Parte 4 resolve.

## 2.1 Composição do índice e tickers necessários

In [116]:
hist_sp500 = pd.read_csv(EXTERNAL_DIR / "sp500_historico.csv")
hist_sp500["date"] = pd.to_datetime(hist_sp500["date"])

def get_constituents(date_str):
    """Tickers do S&P 500 na data indicada (último snapshot ≤ data)."""
    subset = hist_sp500[hist_sp500["date"] <= pd.to_datetime(date_str)]
    return set() if len(subset) == 0 else set(subset.iloc[-1]["tickers"].split(","))

semesters_ext = [
    {"period": f"{y}/S{s}", "year": y, "semester": s,
     "inicio": f"{y}-01-01" if s == 1 else f"{y}-07-01",
     "fim":    f"{y}-06-30" if s == 1 else f"{y}-12-31"}
    for y in range(2016, 2026) for s in (1, 2)
]
todos_tickers = sorted({t for s in semesters_ext for t in get_constituents(s["fim"])})

print(f"Composição: {len(hist_sp500)} snapshots, "
      f"{hist_sp500['date'].min().date()} → {hist_sp500['date'].max().date()}")
print(f"Semestres da extensão: {len(semesters_ext)} (2016/S1 → 2025/S2)")
print(f"Tickers únicos necessários: {len(todos_tickers)}")

Composição: 2705 snapshots, 1996-01-02 → 2026-01-14
Semestres da extensão: 20 (2016/S1 → 2025/S2)
Tickers únicos necessários: 708


## 2.2 Cobertura e coleta

`prices/` guarda o Yahoo, `prices_tiingo/` os delistados e renomeados recuperados pelo
[`gathering.ipynb`](gathering.ipynb). O download só roda se `BAIXAR_FALTANTES = True`,
com checkpoint em `external/coleta_report.csv`.

In [117]:
cobertura = {
    t: "yahoo"  if (PRICES_YAHOO  / f"{t}.csv").exists()
    else "tiingo" if (PRICES_TIINGO / f"{t}.csv").exists()
    else "falta"
    for t in todos_tickers
}
_c = pd.Series(cobertura).value_counts()
print(f"Tickers necessários: {len(todos_tickers)}")
for _k in ("yahoo", "tiingo", "falta"):
    print(f"  {_k:<8}: {_c.get(_k, 0)}")

_faltam = [t for t, v in cobertura.items() if v == "falta"]
if _faltam:
    print(f"\nSem dados: {_faltam}")

if BAIXAR_FALTANTES and _faltam:
    import time
    CHECKPOINT = EXTERNAL_DIR / "coleta_report.csv"
    relatorio  = pd.read_csv(CHECKPOINT) if CHECKPOINT.exists() else \
                 pd.DataFrame(columns=["ticker", "status", "n_rows", "note"])
    for _t in _faltam:
        print(f"  {_t}... ", end="", flush=True)
        try:
            _raw = yf.download(_t, start="2015-07-01", end="2026-01-01",
                               auto_adjust=True, progress=False)
            _cl = _raw[("Close", _t)] if isinstance(_raw.columns, pd.MultiIndex) else _raw["Close"]
            _cl = _cl.rename(_t)
            if len(_cl) > 10:
                _cl.to_csv(PRICES_YAHOO / f"{_t}.csv", header=True)
                _row = {"ticker": _t, "status": "ok", "n_rows": len(_cl), "note": ""}
                cobertura[_t] = "yahoo"
                print(f"ok ({len(_cl)} linhas)")
            else:
                _row = {"ticker": _t, "status": "delisted", "n_rows": len(_cl), "note": "sem dados"}
                print("delisted")
        except Exception as _e:
            _row = {"ticker": _t, "status": "error", "n_rows": 0, "note": str(_e)[:100]}
            print("erro")
        relatorio = pd.concat([relatorio, pd.DataFrame([_row])], ignore_index=True)
        relatorio.to_csv(CHECKPOINT, index=False)
        time.sleep(0.3)
elif _faltam:
    print("\n(BAIXAR_FALTANTES = False — coleta não executada)")

Tickers necessários: 708
  yahoo   : 632
  tiingo  : 76
  falta   : 0


## 2.3 Montagem da extensão

Formato idêntico ao bloco do professor: data no índice, metadados de semestre, um ticker
por coluna, `NaN` fora dos semestres em que o ticker integrava o índice.

**Correção de calendário (`APC`, `INFO`).** Os dois vieram do financialdata.net, única
fonte do projeto que carimba o preço em data de calendário e não de pregão — 224 e 279
datas fora do pregão, das quais 193 e 241 são domingos. Em `APC`, 100% dos domingos têm a
sexta anterior ausente: o valor de sexta foi carimbado no domingo. A correção move cada
data fora do pregão para o pregão anterior e aplica `shift(+1)`. Validado contra o
mercado: `APC` em D vai de +0,005 para +0,459; `INFO` de −0,046 para +0,627.

**Âncora da fronteira.** Os arquivos de `prices/` e `prices_tiingo/` começam em 2015-07,
então já contêm o preço do último pregão do orientador. Guardamos esse valor aqui, **da
mesma série que gera a extensão** — é o que torna a emenda da Parte 4 consistente por
construção. Ver a nota da Parte 4 sobre por que `prices_sobreposição/` não serve como
âncora primária.

In [118]:
# Calendário real de pregão 2016–2025 (o MSFT do Yahoo é a referência)
_msft = pd.read_csv(PRICES_YAHOO / "MSFT.csv", index_col=0, parse_dates=True)
_msft.index = pd.to_datetime(_msft.index, utc=True).tz_convert(None).normalize()
trading_dates_ext = _msft.index[_msft.index >= "2016-01-01"].sort_values()
print(f"Calendário 2016–2025: {len(trading_dates_ext):,} pregões "
      f"({trading_dates_ext[0].date()} → {trading_dates_ext[-1].date()})")

all_prices, ancora_ext, falhas = {}, {}, []
for _t, _src in cobertura.items():
    if _src not in ("yahoo", "tiingo"):
        continue
    try:
        _p = (PRICES_YAHOO if _src == "yahoo" else PRICES_TIINGO) / f"{_t}.csv"
        _df = pd.read_csv(_p, index_col=0, parse_dates=True)
        _df.index = pd.to_datetime(_df.index, utc=True).tz_convert(None).normalize()
        _serie = _df.iloc[:, 0]

        if _t in CALENDARIO_QUEBRADO:
            _cal_set, _idx = set(trading_dates_ext), []
            for _d in _serie.index:
                if _d in _cal_set:
                    _idx.append(_d)
                else:
                    _ant = trading_dates_ext[trading_dates_ext < _d]
                    _idx.append(_ant[-1] if len(_ant) else pd.NaT)
            _serie = pd.Series(_serie.values, index=_idx).dropna()
            _serie = _serie[~_serie.index.duplicated(keep="last")].sort_index().shift(1).dropna()
            print(f"  calendário corrigido: {_t}")

        # Trecho pré-2016 da MESMA série que gera a extensão: é dele que sai a âncora da
        # Parte 4. Guardamos a série inteira (não só a última data) porque cada ticker
        # tem a sua própria data final no bloco do professor.
        _pre = _serie[_serie.index <= DATA_ANCORA]
        if len(_pre):
            ancora_ext[_t] = _pre[(_pre > 0) & np.isfinite(_pre)]

        all_prices[_t] = _serie[_serie.index >= "2016-01-01"]
    except Exception as _e:
        falhas.append((_t, str(_e)[:60]))

print(f"{len(all_prices)} tickers carregados" + (f" | falhas: {falhas}" if falhas else ""))
print(f"Séries com trecho pré-{DATA_ANCORA.year + 1} na própria fonte (âncoras candidatas): {len(ancora_ext)}")

# Metadados de semestre
_meta_rows = []
for _s in semesters_ext:
    _datas = trading_dates_ext[(trading_dates_ext >= _s["inicio"]) & (trading_dates_ext <= _s["fim"])]
    for _j, _d in enumerate(_datas):
        _meta_rows.append({"date": _d, "period": _s["period"], "year": _s["year"],
                           "semester": _s["semester"], "day_in_semester": _j + 1,
                           "total_days_sem": len(_datas)})
meta = pd.DataFrame(_meta_rows).set_index("date")
meta.index = pd.to_datetime(meta.index)

# Máscara de composição do índice
membership = pd.DataFrame(False, index=meta.index, columns=todos_tickers)
for _s in semesters_ext:
    _tk = [t for t in get_constituents(_s["fim"]) if t in membership.columns]
    membership.loc[(meta.index >= _s["inicio"]) & (meta.index <= _s["fim"]), _tk] = True

prices_wide = pd.DataFrame(all_prices).reindex(index=meta.index, columns=todos_tickers)
extensao    = pd.concat([meta[META_COLS], prices_wide.where(membership)], axis=1)

# Trechos interpolados também na extensão (mesmo detector da Parte 1.4)
if REMOVER_TRECHOS_INTERPOLADOS:
    _tot, _n = 0, 0
    for _t in todos_tickers:
        _m = trechos_interpolados(extensao[_t])
        if _m.any():
            extensao.loc[_m.values, _t] = np.nan
            _tot += int(_m.sum()); _n += 1
    print(f"Trechos interpolados removidos da extensão: {_tot:,} células em {_n} séries")

extensao.to_csv(OUTPUT_DIR / "extensao_2016_2025.csv")
print(f"\nextensao_2016_2025.csv: {extensao.shape[0]:,} dias × "
      f"{extensao.shape[1] - len(META_COLS):,} tickers")
print(f"Séries com preço por dia: mediana {extensao[todos_tickers].notna().sum(axis=1).median():.0f}")

Calendário 2016–2025: 2,514 pregões (2016-01-04 → 2025-12-31)
  calendário corrigido: APC
  calendário corrigido: INFO
708 tickers carregados
Séries com trecho pré-2016 na própria fonte (âncoras candidatas): 624
Trechos interpolados removidos da extensão: 273 células em 2 séries

extensao_2016_2025.csv: 2,514 dias × 708 tickers
Séries com preço por dia: mediana 504


---
# Parte 3 — `base_completa.csv` (concatenação direta)

Professor + extensão empilhados, sem tratar a fronteira. O pandas alinha as colunas
sozinho: quem só existe num período fica `NaN` no outro.

Esta é a base **padrão**. Ela carrega o salto de nível na fronteira 2015→2016 descrito na
Parte 4 — serve como controle, não para rodar a estratégia.

In [119]:
base_completa = pd.concat([pt_consolidado, extensao]).sort_index()
_tk_completa  = [c for c in base_completa.columns if c not in META_COLS]

assert not base_completa.index.duplicated().any(), "datas duplicadas na concatenação"

base_completa.to_csv(OUTPUT_DIR / "base_completa.csv")
print(f"Professor  : {pt_consolidado.shape[0]:>5,} dias | {pt_consolidado.shape[1] - len(META_COLS):>4} tickers")
print(f"Extensão   : {extensao.shape[0]:>5,} dias | {extensao.shape[1] - len(META_COLS):>4} tickers")
print(f"base_completa: {base_completa.shape[0]:>5,} dias | {len(_tk_completa):>4} tickers únicos")
print(f"  {base_completa.index[0].date()} → {base_completa.index[-1].date()} "
      f"| {base_completa['period'].nunique()} semestres")

Professor  : 6,426 dias | 1100 tickers
Extensão   : 2,514 dias |  708 tickers
base_completa: 8,940 dias | 1343 tickers únicos
  1990-07-02 → 2025-12-31 | 71 semestres


---
# Parte 4 — `base_unificada.csv` (emenda por retornos)

## O problema

As duas metades usam `adj_close`, mas com **datas de normalização diferentes**: o
orientador coletou em ~dez/2015, nós em 2026. O `adj_close` é retroativo — um split 4:1 da
AAPL em ago/2020 faz o Yahoo dividir por 4 *todo* o histórico. Então o mesmo dia tem preço
diferente conforme *quando* foi baixado, e a concatenação direta cria um salto artificial.

## A solução

Os **retornos diários** são imunes: o fator de ajuste cancela em $P_t/P_{t-1}$. Reescalamos
toda a série da extensão por uma constante por ticker, ancorada no último preço do
orientador:

$$\text{fator} = \frac{P_{\text{prof}}(t_{\text{fim}})}{P_{\text{sobr}}(t_{\text{fim}})}
\qquad\Rightarrow\qquad P_{\text{uni}}(t) = P_{\text{ext}}(t)\times\text{fator}$$

onde $P_{\text{sobr}}$ é o preço de $t_{\text{fim}}$ **na convenção da extensão**. Com isso
o retorno do primeiro dia da extensão vira o retorno real de mercado:

$$r = \frac{P_{\text{ext}}(t_1)\cdot\frac{P_{\text{prof}}(t_{\text{fim}})}{P_{\text{sobr}}(t_{\text{fim}})}}{P_{\text{prof}}(t_{\text{fim}})}-1
    = \frac{P_{\text{ext}}(t_1)}{P_{\text{sobr}}(t_{\text{fim}})}-1$$

Usar $P_{\text{ext}}(t_1)$ no denominador — a versão ingênua — forçaria esse retorno a zero.

## De onde vem a âncora $P_{\text{sobr}}$

Em ordem de preferência:

1. **`ancora_ext`** — o preço de $t_{\text{fim}}$ lido do **próprio arquivo** de `prices/`
   ou `prices_tiingo/` que gera a extensão. Convenção idêntica **por construção**.
2. `prices_sobreposição/` — coleta separada de 2015, usada onde a fonte da extensão não
   alcança $t_{\text{fim}}$.
3. $P_{\text{ext}}(t_1)$ — último recurso, força retorno zero no primeiro dia.

> **Por que `prices_sobreposição/` não é a âncora primária.** Ela foi baixada em data
> diferente de `prices/`, então sofre do mesmo problema que estamos corrigindo, em
> miniatura. Medido: as duas divergem em **34 dos 465** tickers da interseção — desde
> ~0,5% (dividendos pagos entre as duas coletas: `ACN`, `PFE`, `IBM`) até fatores inteiros
> em empresas com spin-off (`DHR` 1,563 pelo spinoff da Fortive; `IR` 0,682 pela Trane;
> `AIV` 0,764 pela Aimco). Usando `prices_sobreposição` como âncora primária, 13 tickers
> saltavam mais de 20% na fronteira; com a âncora da própria fonte isso cai para perto de
> zero (medido em 5.3).

Tickers **só na extensão** (sem âncora) e **só no professor** (delistados) ficam como estão.

In [120]:
prof_tickers = [c for c in pt_consolidado.columns if c not in META_COLS]
ext_tickers  = [c for c in extensao.columns       if c not in META_COLS]
tickers_ambas = sorted(set(prof_tickers) & set(ext_tickers))

print(f"Âncora: {DATA_ANCORA.date()}  |  interseção professor ∩ extensão: {len(tickers_ambas)} tickers")

# Âncora nível 2: prices_sobreposição (coleta separada de 2015)
sobr_ancora = {}
for _t in tickers_ambas:
    _f = PRICES_SOBR / f"{_t}.csv"
    if _f.exists():
        _d = pd.read_csv(_f, index_col=0, parse_dates=[0])
        if DATA_ANCORA in _d.index:
            _v = float(_d.loc[DATA_ANCORA].iloc[0])
            if np.isfinite(_v) and _v != 0:
                sobr_ancora[_t] = _v

cols_rec, fatores, origem, travados = {}, {}, {}, []
for _t in ext_tickers:
    if _t not in tickers_ambas:
        cols_rec[_t] = extensao[_t]; origem[_t] = "sem âncora (só na extensão)"; continue
    _pv, _ev = pt_consolidado[_t].dropna(), extensao[_t].dropna()
    if len(_pv) == 0 or len(_ev) == 0:
        cols_rec[_t] = extensao[_t]; origem[_t] = "sem âncora (série vazia)"; continue

    # A âncora tem de ser lida NA MESMA DATA dos dois lados: t_f é o último pregão em que
    # o bloco do professor tem preço para este ticker, que nem sempre é DATA_ANCORA.
    _tf, _pf = _pv.index[-1], _pv.iloc[-1]
    # O ticker só "emenda" se reaparecer logo depois; se ficou anos fora do índice, não há
    # fronteira contínua para conferir — e o primeiro preço da extensão não é comparável.
    _contiguo = (_ev.index[0] - _tf).days <= 10

    _a = ancora_ext.get(_t)
    if _a is not None and _tf in _a.index:          # 1. fonte da própria extensão
        _fator, _o = _pf / float(_a.loc[_tf]), "fonte da extensão"
    elif _tf == DATA_ANCORA and _t in sobr_ancora:  # 2. prices_sobreposição
        _fator, _o = _pf / sobr_ancora[_t], "prices_sobreposição"
    elif _contiguo:                                 # 3. retorno zero no 1º dia
        _fator, _o = _pf / _ev.iloc[0], "fallback (retorno zero)"
    else:                                           # 4. sem âncora e sem fronteira
        cols_rec[_t] = extensao[_t]; origem[_t] = "sem âncora (reentrada tardia)"; continue

    # Trava: se a âncora implicar um salto implausível na fronteira, ela está em outra
    # convenção (spin-off, troca de ticker recuperada de fonte distinta). Cai para o
    # fallback — fabricado, mas limitado, em vez de um salto de −40%.
    if _contiguo:
        _r1 = _ev.iloc[0] * _fator / _pf - 1
        if abs(_r1) > LIMITE_SALTO_FRONTEIRA:
            travados.append({"ticker": _t, "ancora": _o, "retorno_implicado": _r1,
                             "ultimo_dia_prof": str(_tf.date())})
            _fator, _o = _pf / _ev.iloc[0], "travado → fallback"

    fatores[_t], origem[_t] = _fator, _o
    cols_rec[_t] = extensao[_t] * _fator

print("\nOrigem da âncora:")
for _k, _v in pd.Series(origem).value_counts().items():
    print(f"  {_k:<30} {_v:>4}")
_fb = [t for t, o in origem.items() if o == "fallback (retorno zero)"]
if _fb:
    print(f"  → fallback puro: {_fb}")
if travados:
    print(f"\nTravados pelo LIMITE_SALTO_FRONTEIRA ({LIMITE_SALTO_FRONTEIRA:.0%}):")
    print(pd.DataFrame(travados).sort_values("retorno_implicado").to_string(index=False))

ext_rec = pd.concat([extensao[META_COLS], pd.DataFrame(cols_rec)], axis=1)
base_unificada = pd.concat([pt_consolidado, ext_rec]).sort_index()

assert not base_unificada.index.duplicated().any(), "datas duplicadas"
base_unificada.to_csv(OUTPUT_DIR / "base_unificada.csv")

_fs = pd.Series(fatores)
print(f"\nFator de escala: mediana {_fs.median():.4f} | "
      f"min {_fs.min():.4f} ({_fs.idxmin()}) | máx {_fs.max():.4f} ({_fs.idxmax()})")
print(f"  |fator − 1| > 30%: {((_fs - 1).abs() > 0.3).sum()} tickers (splits e dividendos pós-2015)")
print(f"\nbase_unificada.csv: {base_unificada.shape[0]:,} dias × "
      f"{len([c for c in base_unificada.columns if c not in META_COLS]):,} tickers")
print(f"  {base_unificada.index[0].date()} → {base_unificada.index[-1].date()}")

Âncora: 2015-12-31  |  interseção professor ∩ extensão: 465 tickers

Origem da âncora:
  fonte da extensão               420
  sem âncora (só na extensão)     243
  prices_sobreposição              26
  sem âncora (reentrada tardia)    12
  travado → fallback                7

Travados pelo LIMITE_SALTO_FRONTEIRA (25%):
ticker              ancora  retorno_implicado ultimo_dia_prof
   CHK prices_sobreposição          -0.994500      2015-12-31
   HCP prices_sobreposição          -0.415681      2015-12-31
   STI prices_sobreposição          -0.409718      2015-12-31
    LB prices_sobreposição          -0.390424      2015-12-31
   ADS prices_sobreposição          -0.318516      2015-12-31
   GPS   fonte da extensão          -0.278186      2015-12-31
   COG prices_sobreposição          -0.274873      2015-12-31

Fator de escala: mediana 1.2774 | min 0.2365 (GE) | máx 50.0000 (CMG)
  |fator − 1| > 30%: 213 tickers (splits e dividendos pós-2015)

base_unificada.csv: 8,940 dias × 1,343 tickers

---
# Parte 5 — Validação

Cinco verificações numéricas. Nenhuma depende de inspeção visual.

## 5.1 Sanidade das duas bases

In [121]:
for _nome, _b in [("base_completa", base_completa), ("base_unificada", base_unificada)]:
    _tk = [c for c in _b.columns if c not in META_COLS]
    _v  = _b[_tk]
    print(f"{_nome:<16} {_b.shape[0]:>6,} dias × {len(_tk):>5} tickers | "
          f"zeros {int((_v == 0).sum().sum()):>3} | negativos {int((_v < 0).sum().sum()):>3} | "
          f"datas duplicadas {int(_b.index.duplicated().sum())} | "
          f"séries/dia mediana {_v.notna().sum(axis=1).median():.0f}")

base_completa     8,940 dias ×  1343 tickers | zeros   0 | negativos   0 | datas duplicadas 0 | séries/dia mediana 499
base_unificada    8,940 dias ×  1343 tickers | zeros   0 | negativos   0 | datas duplicadas 0 | séries/dia mediana 499


## 5.2 A emenda preserva os retornos da extensão

`ext_rec = ext × k` com `k` constante por ticker, então por álgebra
`r_rec(t) = r_ext(t)`. O teste mede o desvio numérico — deve ser da ordem da precisão de
máquina.

In [122]:
_corr, _maxd = {}, {}
for _t in tickers_ambas:
    _a, _b = extensao[_t].dropna(), ext_rec[_t].dropna()
    _i = _a.index.intersection(_b.index)
    if len(_i) < 10:
        continue
    _ra, _rb = _a.loc[_i].pct_change().dropna(), _b.loc[_i].pct_change().dropna()
    _corr[_t] = _ra.corr(_rb)
    _maxd[_t] = (_ra - _rb).abs().max()

_cs, _ms = pd.Series(_corr), pd.Series(_maxd)
print(f"Tickers testados          : {len(_cs)}")
print(f"Correlação mínima         : {_cs.min():.10f}  ({_cs.idxmin()})")
print(f"Tickers com corr < 0.9999 : {(_cs < 0.9999).sum()}")
print(f"max |r_ext − r_rec| global: {_ms.max():.2e}")
print(f"Tickers com desvio > 1e-8 : {(_ms > 1e-8).sum()}")

Tickers testados          : 465
Correlação mínima         : 1.0000000000  (CSX)
Tickers com corr < 0.9999 : 0
max |r_ext − r_rec| global: 2.22e-16
Tickers com desvio > 1e-8 : 0


## 5.3 A fronteira deixou de ter salto

Compara o retorno do **primeiro dia da extensão** nas duas bases. Em `base_completa` ele
carrega o fator de split (salto artificial); em `base_unificada` deve ser um retorno de
mercado normal — da mesma ordem dos demais dias.

In [123]:
_d1 = extensao.index[0]                                   # primeiro pregão da extensão
_tk = [t for t in tickers_ambas if t in base_completa.columns]

_r_comp = base_completa[_tk].pct_change(fill_method=None).loc[_d1]
_r_uni  = base_unificada[_tk].pct_change(fill_method=None).loc[_d1]
_r_tip  = base_unificada[_tk].pct_change(fill_method=None).loc["2016-01-05":"2016-06-30"].abs().median().median()

print(f"Primeiro pregão da extensão: {_d1.date()}   ({len(_tk)} tickers na interseção)")
print(f"{'':<18}{'|r| mediano':>14}{'|r| > 20%':>12}{'|r| máximo':>14}")
for _nm, _r in [("base_completa", _r_comp), ("base_unificada", _r_uni)]:
    print(f"{_nm:<18}{_r.abs().median():>14.4f}{int((_r.abs() > 0.2).sum()):>12}{_r.abs().max():>14.4f}")
print(f"\nReferência: |r| diário mediano no resto de 2016/S1 = {_r_tip:.4f}")
print("A base_unificada deve ficar na ordem da referência; a base_completa, muito acima.")

_res = _r_uni[_r_uni.abs() > 0.2].sort_values()
if len(_res):
    print(f"\nResíduos na base_unificada ({len(_res)}) — conferir origem da âncora:")
    print(pd.DataFrame({"retorno": _res.round(4),
                        "ancora": [origem.get(t, "-") for t in _res.index]}).to_string())
else:
    print("\nNenhum ticker salta mais de 20% na fronteira da base_unificada.")

Primeiro pregão da extensão: 2016-01-04   (465 tickers na interseção)
                     |r| mediano   |r| > 20%    |r| máximo
base_completa             0.2285         253        3.1690
base_unificada            0.0157           1        0.2468

Referência: |r| diário mediano no resto de 2016/S1 = 0.0093
A base_unificada deve ficar na ordem da referência; a base_completa, muito acima.

Resíduos na base_unificada (1) — conferir origem da âncora:
     retorno               ancora
ABC  -0.2468  prices_sobreposição


## 5.4 Alinhamento de datas nos dois blocos

Correlação do retorno de cada série com o retorno mediano do mercado, nos lags D−1, D e
D+1. Séries alinhadas têm o pico em D. É o teste que detectou as 5 séries deslocadas.

In [124]:
_R   = base_unificada[[c for c in base_unificada.columns if c not in META_COLS]].pct_change(fill_method=None)
_mkt = _R.loc[:, _R.notna().sum() > 250].median(axis=1)

for _nm, _m in [("professor 1990–2015", base_unificada.index <= DATA_ANCORA),
                ("extensão  2016–2025", base_unificada.index >  DATA_ANCORA)]:
    _sub = _R.loc[_m]
    _mk  = _mkt.loc[_m]
    _ok  = _sub.columns[_sub.notna().sum() > 250]
    _pico = []
    for _t in _ok:
        _v = [_sub[_t].corr(_mk.shift(1)), _sub[_t].corr(_mk), _sub[_t].corr(_mk.shift(-1))]
        if not np.all(np.isnan(_v)):
            _pico.append(["D-1", "D", "D+1"][int(np.nanargmax(_v))])
    _p = pd.Series(_pico).value_counts()
    _beta = np.nanmedian([_sub[_t].corr(_mk) for _t in _ok])
    print(f"{_nm}: {len(_ok)} séries | pico em D: {_p.get('D', 0)} "
          f"| D-1: {_p.get('D-1', 0)} | D+1: {_p.get('D+1', 0)} | corr mediana em D: {_beta:.3f}")
print("\nEsperado: quase todas em D. Séries curtas e ilíquidas geram algum ruído.")

professor 1990–2015: 1053 séries | pico em D: 1048 | D-1: 4 | D+1: 1 | corr mediana em D: 0.492
extensão  2016–2025: 679 séries | pico em D: 679 | D-1: 0 | D+1: 0 | corr mediana em D: 0.580

Esperado: quase todas em D. Séries curtas e ilíquidas geram algum ruído.


## 5.5 Contra download independente

Baixa a série completa 1990–2025 do Yahoo para tickers com splits relevantes **depois**
de 2015 e compara os retornos diários com a `base_unificada`, antes e depois da fronteira.

O `adj_close` é retroativo, então o fator de ajuste cancela em retornos de dois dias
consecutivos — a correlação deve ser ≈ 1 nos **dois** períodos.

| Ticker | Splits pós-2015 |
|---|---|
| `AAPL` | 4:1 ago/2020 |
| `AMZN` | 20:1 jun/2022 |
| `GOOGL` | 20:1 jul/2022 |
| `NVDA` | 4:1 set/2021, 10:1 jun/2024 |
| `MSFT` | — (só dividendos) |

In [125]:
_TKF = ["AAPL", "AMZN", "GOOGL", "NVDA", "MSFT"]
_raw = yf.download(_TKF, start="1990-01-01", end="2026-01-01", auto_adjust=True, progress=False)["Close"]
_raw.index = pd.to_datetime(_raw.index).tz_localize(None)

print(f"{'Ticker':<8}{'Período':<12}{'N obs':>8}{'Correlação':>14}{'max |Δr|':>12}")
print("-" * 54)
for _t in _TKF:
    if _t not in base_unificada.columns or _t not in _raw.columns:
        continue
    _u, _w = base_unificada[_t].dropna(), _raw[_t].dropna()
    _i = _u.index.intersection(_w.index)
    _ru, _rw = _u.loc[_i].pct_change().dropna(), _w.loc[_i].pct_change().dropna()
    _ir = _ru.index.intersection(_rw.index)
    _ru, _rw = _ru.loc[_ir], _rw.loc[_ir]
    for _lb, _m in [("completo", pd.Series(True, index=_ir)),
                    ("pré-2016", _ir <= DATA_ANCORA),
                    ("pós-2016", _ir >  DATA_ANCORA)]:
        if _m.sum() < 10:
            continue
        print(f"{_t:<8}{_lb:<12}{int(_m.sum()):>8,}"
              f"{_ru[_m].corr(_rw[_m]):>14.8f}{(_ru[_m] - _rw[_m]).abs().max():>12.2e}")
    print()

Ticker  Período        N obs    Correlação    max |Δr|
------------------------------------------------------
AAPL    completo       8,939    0.99953528    1.41e-02
AAPL    pré-2016       6,425    0.99946433    1.41e-02
AAPL    pós-2016       2,514    1.00000000    1.33e-06

AMZN    completo       5,029    0.99941485    3.04e-02
AMZN    pré-2016       2,515    0.99906242    3.04e-02
AMZN    pós-2016       2,514    1.00000000    4.44e-16

GOOGL   completo       4,904    0.99988342    5.17e-03
GOOGL   pré-2016       2,390    0.99977387    5.17e-03
GOOGL   pós-2016       2,514    1.00000000    6.03e-07

NVDA    completo       6,037    0.99966086    2.31e-02
NVDA    pré-2016       3,523    0.99946242    2.31e-02
NVDA    pós-2016       2,514    1.00000000    9.75e-07

MSFT    completo       7,927    0.99923115    2.42e-02
MSFT    pré-2016       5,413    0.99898981    2.42e-02
MSFT    pós-2016       2,514    1.00000000    1.01e-06



---
## Saídas

| Arquivo | O que é |
|---|---|
| `professor_consolidado.csv` | bloco do orientador com datas e correções (intermediário) |
| `extensao_2016_2025.csv` | extensão própria 2016–2025 (intermediário) |
| `base_completa.csv` | concatenação direta — **padrão**, com o salto na fronteira |
| **`base_unificada.csv`** | extensão emendada por retornos — **use esta** |

Próximos passos: [`serie_rf.ipynb`](serie_rf.ipynb) para a taxa livre de risco, depois
`distance/pairs_formation.ipynb` e `distance/pairs_trading.ipynb`.